### Chapter 7: Search In-Depth
### Topic 8: Metadata Filtering as a Search-Time Tool

### Metadata Filtering

### Why Does This Exist?

Suppose your company has stored **10 lakh document chunks** inside Qdrant.

Examples:

```text
FD FAQs
Loan FAQs
Insurance FAQs
Credit Card FAQs
HR Policies
Internal Documents
```

Now a customer asks:

> "What is the penalty for premature FD withdrawal?"

Dense Retrieval or BM25 understands only the **text** of the query.

It does **not** know things like:

- Search only FD documents.
- Search only English documents.
- Search only documents uploaded this year.
- Search only Bajaj Finance documents.
- Search only documents belonging to Customer A.

These are **structured conditions**, not part of the natural language query.

This is why **metadata filtering** exists.

---

### What Is Metadata?

Metadata is simply **information about a document**, not the document itself.

Example:

```text
Document

Premature withdrawal of FD incurs a 1% penalty.
```

Metadata stored with it:

```text
Category      : FD
Language      : English
Department    : Retail Banking
Product       : Fixed Deposit
Uploaded Date : 2025-03-15
Owner          : Bajaj Finance
```

The embedding is created only from the document text.

The metadata is stored separately.

---

### Why Can't Embeddings Solve This?

Suppose the user asks:

> "Show FD documents uploaded in 2025."

The embedding model only understands the sentence.

It does **not** understand that

- 2025 is a filter
- FD is a category restriction
- Uploaded Date is a database field

It simply converts text into vectors.

Therefore, embeddings cannot enforce structured constraints.

---

### What Does Metadata Filtering Do?

Metadata filtering tells the vector database:

> "Before searching, only consider documents whose metadata satisfies these conditions."

Example:

```text
Category = FD

Language = English

Uploaded Year = 2025
```

Now semantic search happens only on those documents.

---

### Why Is This Important?

Without filtering, the retriever searches the entire knowledge base.

```text
Knowledge Base

FD
Loans
Insurance
HR
Credit Cards
```

Even if the customer only wants FD information.

Filtering reduces the search space.

Benefits:

- Better relevance
- Faster search
- Lower LLM cost
- Fewer incorrect documents

---

### Why Is It Even More Important for AI Agents?

Humans often assume context.

Suppose an employee asks:

> "Show the latest policy."

They already know they mean:

- HR policy
- Internal document
- Latest version

But the query never says that.

An AI Agent can infer this context and automatically apply filters like:

```text
Department = HR

Document Type = Policy

Latest Version = True
```

The user never has to specify them.

The agent decides the filters and sends them to Qdrant.

---

### Two Ways Filtering Can Be Applied

Metadata filtering can happen in two different ways.

The difference is very important.

---

### Post-Filtering

First, semantic search is performed across **all documents**.

Example:

```text
Top Results

1. Loan Policy
2. Insurance FAQ
3. FD FAQ
4. HR Policy
5. FD Charges
```

Now the filter is applied.

Example:

```text
Category = FD
```

After filtering:

```text
3. FD FAQ

5. FD Charges
```

The other results are discarded.

Problem:

If the top results contain very few FD documents, the final result may contain only one or two documents, even though many better FD documents existed deeper in the database.

---

### Pre-Filtering (Filter Pushdown)

The metadata filter is applied **before similarity search begins**.

Example:

```text
Category = FD
```

Now the search runs **only** on FD documents.

Results:

```text
1. FD FAQ

2. FD Charges

3. FD Premature Withdrawal

4. FD Interest Rules

5. FD Closure Process
```

The retriever never wastes effort searching Loan or Insurance documents.

This usually produces much better results.

---

### Why Is Pre-Filtering Better?

Because semantic search happens only on documents that already satisfy the business rules.

Benefits:

- Higher relevance
- Faster retrieval
- Better recall
- Lower computation
- Better production performance

Most production vector databases like **Qdrant** perform **filter pushdown**, making pre-filtering the preferred approach.

---

### Internal Working — Step by Step

**Post-filter mechanics (the naive, often-wrong approach):**

1. Embed the query.
2. Run similarity search against every stored vector, regardless of metadata, and get the top-k results.
3. Discard any of those top-k results that don't match the desired metadata filter.
4. Return whatever remains — which could be fewer than k results, sometimes even zero.

- The critical flaw: if most of the corpus's most-similar vectors happen to belong to a category that gets filtered out, the top-k *before* filtering may end up containing zero matching documents at all — and post-filtering then returns nothing, even though genuinely matching documents exist elsewhere in the corpus, just not within that initial unfiltered top-k.

**Pre-filter / filter-pushdown mechanics (the correct approach for most use cases):**

1. Embed the query.
2. During the search process itself, only consider vectors whose metadata actually matches the filter — non-matching vectors are never even compared to the query.
3. Return the top-k among only the matching subset.

- Most modern vector databases support this natively: the filter condition is passed alongside the query vector and applied *during* the search traversal, not after.
- This guarantees that if k matching documents exist anywhere in the collection, and they're even moderately similar to the query, they will actually be found — the filter narrows the search space itself, rather than trimming down a result set after the fact.

**Combining filtering with everything built in earlier topics:**

- For BM25 (Topic 1): metadata filtering can be applied by restricting which documents are included in the scanned corpus before scoring even happens.
- For dense retrieval (Topic 2) via a vector database: native pre-filter support, as described above.
- For hybrid RRF (Topic 4): the *same* metadata filter must be applied to both the BM25 call and the dense retrieval call *before* fusion — applying it inconsistently (filtering one retriever's output but not the other's) is a correctness bug, and can even become a data-leak risk if the filter is meant to enforce access control.
- For reranking (Topic 7): filtering should already have happened upstream by the time candidates reach the reranker — reranking a candidate pool that still contains filtered-out documents wastes reranking compute on documents that will never actually be returned anyway.

---

### How This Is Implemented in This Project

- Every chunk in the knowledge base carries metadata fields like which source document it came from and a categorical content type (for example, FAQ, policy, procedure, or product information).
- A realistic search-time filtering scenario: a customer asks a question that an earlier classification step has already determined is specifically about a procedure (like "what documents do I need to submit to close my FD early?") rather than a general FAQ-style question. Rather than searching the entire knowledge base, the retrieval call can pre-filter to only the procedural content type, narrowing the search space to exactly the content most likely to contain the answer, and reducing the chance that a superficially similar FAQ or policy chunk outranks the actually-correct procedural chunk.
- The implementation pattern: apply the exact same pre-filter to both the dense retrieval call and the BM25 call before combining their results — never filter one and not the other, since that would create inconsistent, potentially incorrect or leaky results.

---


### Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **The post-filter "fewer than k results" failure, precisely stated:** if a query's true top-20 nearest neighbors (before filtering) happen to contain only 2 documents matching the desired category, post-filtering to that category after search returns only those 2 results — even if 15 genuinely matching documents exist elsewhere in the collection with moderately lower but still relevant similarity scores. Pre-filtering avoids this entirely by searching only within the matching subset from the very start, so the returned top-k is genuinely the best k among actual matches, not the best k overall with non-matches discarded afterward.
- **Filter selectivity is a genuine trade-off, not just a bug to avoid:** a highly selective filter (restricting to a very small subset of the corpus) narrows the search space significantly — great for precision if the filter is correct, but if the filter itself is wrong (the true answer is actually somewhere else entirely), pre-filtering guarantees the correct document can never be found, no matter how semantically similar it might be. This is the central real-time dilemma of this topic: filtering trades recall-if-the-filter-is-wrong for precision-if-the-filter-is-right — the correctness of the filter itself becomes a new potential failure mode that didn't exist before filtering was introduced.
- **Latency:** pre-filtering during search has a modest cost compared to unfiltered search, since the search process must additionally check the metadata condition at each step, but for smaller corpora this overhead is negligible. At much larger scale, highly selective filters can sometimes make the underlying search index less efficient than expected, because the index structure was built for the full space, not the filtered subspace — dedicated payload/metadata indexes exist specifically to address this, but it's worth knowing this isn't free at scale.
- **Consistency across retrievers in a hybrid setup:** applying a filter to only one of two retrievers being fused is a real, easy-to-introduce bug — it can silently make results inconsistent, or in an access-control context, actually leak records that should have been filtered out on both sides.
- **Debugging empty or too-few results:** first check whether the search was pre-filtered or post-filtered — if post-filtered, the empty result might simply mean matching documents exist but weren't in the initial unfiltered top-k, which pre-filtering would have caught. If genuinely pre-filtered and still empty, either no matching documents exist at all, or the filter value itself might be wrong (a typo in a category name, for example).
- **Monitoring:** track how often filtered searches return fewer results than requested — a rising trend can indicate either the filter values are too restrictive for the actual content available, or that content coverage for a specific category has gaps that need to be filled.
- **Security:** any filter used for access control (like restricting results to a specific customer's own records) must be applied consistently and pre-filter style across every retrieval path being used — a post-filter approach used for access control is a real security risk, since it means unauthorized records were still fetched and processed internally before being discarded, and any bug in the discarding logic becomes a data leak.

---


### Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Pre-filter vs post-filter:** pre-filtering is correct for almost every use case that involves accuracy-sensitive results or any access-control requirement, since post-filtering can silently return fewer results than requested or leak unauthorized records if a discard step has a bug. Post-filtering might be acceptable only in low-stakes situations where occasionally getting fewer results than requested is a genuinely acceptable trade-off for simpler implementation.
- **How selective to make a filter:** a highly selective filter improves precision when it's correct, but completely eliminates the chance of finding the right document if the filter itself is wrong. This means filter selectivity should be chosen based on how confident the upstream process is that the filter value is actually correct — an uncertain or weakly-confident classification shouldn't drive an aggressively narrow filter.
- **Who decides the filter value:** a filter value can be manually specified by a human, derived deterministically from some other signal (like an earlier classification step), or decided dynamically by an intelligent agent reasoning about the query. Each source of the filter value carries a different level of confidence, which should inform how aggressively that filter is applied.

---


### Alternatives and When to Use Each

- **Pre-filtering (filter-pushdown):** the correct default for almost all cases — use whenever result completeness matters, or whenever the filter serves any access-control purpose.
- **Post-filtering:** acceptable only for low-stakes exploratory use cases where getting occasionally fewer results than requested is a genuinely acceptable trade-off in exchange for simpler implementation, and where there's no access-control requirement involved.
- **No filtering at all (broad, unfiltered search):** appropriate when there's no reliable structured signal available to filter on, or when the upstream confidence in any candidate filter value is too low to justify potentially eliminating the correct answer from consideration entirely.
- **Soft filtering / boosting instead of hard filtering:** an alternative worth considering when filter confidence is moderate but not certain — instead of eliminating non-matching documents entirely, boost the ranking of matching documents while still allowing non-matching ones to be found if they're strongly relevant. This avoids the all-or-nothing risk of a hard pre-filter being wrong.

---


### Common Mistakes and Production Failures

- Using post-filtering for a use case where completeness of results actually matters, and not realizing that this can silently return far fewer results than requested — or none at all — even when matching documents genuinely exist in the collection.
- Applying a metadata filter to only one retriever in a hybrid retrieval setup, creating inconsistent results, or in an access-control context, a real data-leak risk.
- Using post-filtering for access control specifically — this means unauthorized records were still internally fetched and processed before being discarded, and any bug in that discarding logic becomes an actual security incident, not just a quality issue.
- Applying an overly aggressive, highly selective filter based on a low-confidence upstream signal, which can eliminate the correct answer from consideration entirely if that upstream signal happens to be wrong.
- Not accounting for the performance cost of highly selective filters at large scale, assuming filtering is always free just because it appeared negligible at small scale during development.

---


### Lead-Level Interview Questions

**Basic**

- Q: What's the difference between pre-filtering and post-filtering in retrieval?  
  A: Pre-filtering applies the metadata constraint during the search process itself, so only matching documents are ever considered. Post-filtering runs the full search first and then discards non-matching results afterward — which means matching documents that weren't in the unfiltered top results are permanently missed, even if they exist elsewhere in the collection.

- Q: Why can post-filtering return fewer results than requested, even when matching documents exist?  
  A: Because the initial unfiltered search only returns a limited number of top results before any filtering happens. If most of those top results don't match the filter, there's nothing left after filtering — even though other, slightly less similar but still relevant matching documents exist further down in the full collection, outside that initial unfiltered top-k.

**Intermediate**

- Q: What's the central trade-off introduced by adding a metadata filter to a retrieval query?  
  A: Filtering trades recall-if-the-filter-is-wrong for precision-if-the-filter-is-right. A correct filter meaningfully improves precision by removing irrelevant categories of content from consideration. But if the filter value itself is incorrect, it guarantees the correct answer can never be found, no matter how semantically relevant it actually is — the filter's own correctness becomes a new failure mode that didn't exist before filtering was introduced.

- Q: In a hybrid retrieval setup combining two different retrieval methods, why is filter consistency important?  
  A: If a metadata filter is applied to one retriever's results but not the other's, the two ranked lists being combined are no longer operating over the same effective search space, producing inconsistent or incorrect final results. In an access-control context, this inconsistency can actually leak unauthorized records through the unfiltered retriever's path.

**Advanced**

- Q: A teammate implements access control by fetching all results first and then filtering out records the caller isn't authorized to see. What's wrong with this approach?  
  A: This is post-filtering used for access control, which is a genuine security risk — unauthorized records were still internally fetched and processed before being discarded, meaning they passed through more of the system than they should have, and any bug in the discard logic (or any code path that accesses results before filtering happens) becomes an actual data leak, not just a quality problem. Access control filters should always be applied as pre-filters, during the search itself, so unauthorized records are never fetched in the first place.

- Q: How would you decide whether to apply a highly selective filter versus a broader, unfiltered search for a given query?  
  A: Base it on the confidence of whatever signal is producing the filter value. If the filter comes from a highly reliable, well-validated upstream classification, a selective filter can meaningfully improve precision. If the filter comes from an uncertain or weak signal, an aggressive filter risks eliminating the correct answer entirely if that signal turns out to be wrong — in that case, either a broader search or a soft-boosting approach (favoring but not requiring the filter match) is safer.

**Scenario-based**

- Q: Users report that filtered searches for a specific content category are frequently returning zero or very few results, even though that content clearly exists in the knowledge base. Walk through your diagnosis.  
  A: First confirm whether the filtering is implemented as pre-filter or post-filter — if it's post-filter, this exact symptom (matching content existing but not appearing in results) is the expected, known failure mode, and switching to pre-filtering should resolve it directly. If it's already pre-filtered and still returning too few results, check whether the filter value itself is correct (a typo or mismatched category name would produce this exact symptom too), and check whether the actual content coverage for that category is as complete as assumed.

**Follow-up questions to expect:**

- "How would you monitor whether your filtering strategy is helping or hurting overall retrieval quality?"
- "What would you do differently if the filter value was being decided by an automated upstream process rather than a human?"

---


### Hidden Concepts / Prerequisites Worth Knowing

- **Metadata filtering as one instance of a general pattern:** combining unstructured similarity search with structured constraints is a pattern that shows up broadly, anywhere search needs to respect both "what does this content mean" and "what category or property must this content have" — recognizing metadata filtering as an instance of this general pattern, rather than a narrow feature unique to one system, is a strong signal of genuine understanding.
- **Payload/metadata indexes as a scaling mechanism:** at large scale, dedicated indexes over the metadata fields themselves (separate from the main similarity index) are what make pre-filtering efficient — without them, checking the filter condition for every candidate during traversal can become a real performance cost.
- **The relationship between filter confidence and filter aggressiveness:** this is really a specific instance of a much more general decision-making principle — the more confident you are in a piece of information, the more aggressively you can act on it; the less confident, the more you should hedge. Applying this lens to filter selectivity reveals why an uncertain upstream classification shouldn't drive an aggressively narrow filter.

--- 

### Quick Revision Summary (for last-minute interview prep)

> Metadata filtering lets structured constraints (like content category, source, or ownership) be combined with unstructured semantic or lexical search, addressing a real gap: retrieval methods built purely on text similarity have no native way to represent structural requirements a human or an intelligent agent might have in mind. The critical technical distinction is pre-filtering (applying the constraint during search, so only matching documents are ever considered) versus post-filtering (searching everything first, then discarding non-matches afterward) — post-filtering can silently return far fewer results than requested, or even zero, when matching documents exist but simply weren't in the initial unfiltered top results. Pre-filtering should always be used when result completeness matters or when the filter serves any access-control purpose, since post-filtering for access control means unauthorized records were still internally processed before being discarded — a real security risk, not just a quality issue. In any hybrid retrieval setup, the same filter must be applied consistently across every retrieval method being combined. And fundamentally, adding a filter trades recall-if-wrong for precision-if-right — a highly selective filter based on an uncertain upstream signal can eliminate the correct answer from consideration entirely if that signal turns out to be mistaken.

---


### Module 1: Setup — A Corpus Where Filtering Actually Matters

Build a knowledge base where the truly relevant documents for a specific category are NOT among the overall top-similarity results — this is the exact scenario where post-filtering fails and pre-filtering succeeds.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- a corpus where post-filtering will fail
#
# Deliberately built so that most of the HIGHEST similarity documents
# for our test query belong to doc_type "faq", while the genuinely
# correct answers for a "sop"-specific query are only moderately
# similar overall -- exactly the scenario that breaks post-filtering.
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

KNOWLEDGE_BASE = [
    {"id": 0, "doc_type": "faq",    "text": "What is the penalty for premature FD withdrawal? It is 1 percent."},
    {"id": 1, "doc_type": "faq",    "text": "What is the FD interest rate for 24 months? It is 7.1 percent."},
    {"id": 2, "doc_type": "faq",    "text": "Can senior citizens get extra FD interest? Yes, an additional 0.5 percent."},
    {"id": 3, "doc_type": "faq",    "text": "Can I submit a request to close my FD account online instead of visiting a branch?"},
    {"id": 4, "doc_type": "sop",    "text": "Step 1 of FD closure: submit a written closure request form to the branch."},
    {"id": 5, "doc_type": "sop",    "text": "Step 2 of FD closure: attach a copy of the passbook and identity proof."},
    {"id": 6, "doc_type": "policy", "text": "Fixed Deposit closure requires RBI-compliant documentation per policy."},
]

# a query about the SOP-specific procedure -- but phrased in a way that
# overlaps heavily with the FAQ documents' vocabulary ("FD", "closure")
QUERY = "what do I need to submit to close my FD"
TARGET_DOC_TYPE = "sop"

def normalize_vector(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

texts = [d["text"] for d in KNOWLEDGE_BASE]
vectorizer = TfidfVectorizer()
sparse_matrix = vectorizer.fit_transform(texts)
svd = TruncatedSVD(n_components=4, random_state=42)
doc_vectors = svd.fit_transform(sparse_matrix)
doc_vectors_norm = np.array([normalize_vector(v) for v in doc_vectors])

query_vector = normalize_vector(svd.transform(vectorizer.transform([QUERY]))[0])
all_scores = np.array([cosine_similarity(query_vector, v) for v in doc_vectors_norm])
overall_ranking = np.argsort(all_scores)[::-1]

print("=" * 70)
print("OVERALL SIMILARITY RANKING (no filtering applied)")
print("=" * 70)
print(f"Query: '{QUERY}'\n")
for rank, idx in enumerate(overall_ranking, start=1):
    doc = KNOWLEDGE_BASE[idx]
    print(f"  Rank {rank} | Doc {idx} [{doc['doc_type']:<6}] | score={all_scores[idx]:.4f} | {doc['text'][:55]}...")

sop_indices = [i for i, d in enumerate(KNOWLEDGE_BASE) if d["doc_type"] == TARGET_DOC_TYPE]
sop_ranks = [list(overall_ranking).index(i) + 1 for i in sop_indices]
print(f"\nThe actual SOP documents (Doc {sop_indices}) rank at overall")
print(f"positions {sop_ranks} out of {len(KNOWLEDGE_BASE)} -- notice they are")
print("NOT necessarily in the overall top few, even though they are the")
print("genuinely correct answer for this procedure-specific question.")
print("\nModule 1 complete. Run Module 2 next.")


OVERALL SIMILARITY RANKING (no filtering applied)
Query: 'what do I need to submit to close my FD'

  Rank 1 | Doc 3 [faq   ] | score=0.9612 | Can I submit a request to close my FD account online in...
  Rank 2 | Doc 4 [sop   ] | score=0.8656 | Step 1 of FD closure: submit a written closure request ...
  Rank 3 | Doc 5 [sop   ] | score=0.7281 | Step 2 of FD closure: attach a copy of the passbook and...
  Rank 4 | Doc 2 [faq   ] | score=0.3626 | Can senior citizens get extra FD interest? Yes, an addi...
  Rank 5 | Doc 1 [faq   ] | score=0.2394 | What is the FD interest rate for 24 months? It is 7.1 p...
  Rank 6 | Doc 0 [faq   ] | score=0.2141 | What is the penalty for premature FD withdrawal? It is ...
  Rank 7 | Doc 6 [policy] | score=-0.1531 | Fixed Deposit closure requires RBI-compliant documentat...

The actual SOP documents (Doc [4, 5]) rank at overall
positions [2, 3] out of 7 -- notice they are
NOT necessarily in the overall top few, even though they are the
genuinely correct an

### Module 2: Post-Filter vs Pre-Filter, Side by Side

Runs both strategies on the exact same query and corpus, showing the concrete difference in results.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Post-filter vs pre-filter -- concrete comparison
# ------------------------------------------------------------------

def post_filter_search(query_vector, doc_vectors, knowledge_base,
                        doc_type: str, top_k_before_filter: int = 3, final_k: int = 3):
    """WRONG approach for correctness-sensitive use cases: search first
    over EVERYTHING, take the top few, THEN discard non-matching ones."""
    scores = np.array([cosine_similarity(query_vector, v) for v in doc_vectors])
    top_indices = np.argsort(scores)[::-1][:top_k_before_filter]
    filtered = [i for i in top_indices if knowledge_base[i]["doc_type"] == doc_type]
    return filtered[:final_k]


def pre_filter_search(query_vector, doc_vectors, knowledge_base,
                       doc_type: str, final_k: int = 3):
    """CORRECT approach: restrict to matching documents FIRST, then rank
    only within that matching subset."""
    matching_indices = [i for i, d in enumerate(knowledge_base) if d["doc_type"] == doc_type]
    if not matching_indices:
        return []
    scores = [(i, cosine_similarity(query_vector, doc_vectors[i])) for i in matching_indices]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [i for i, _ in scores[:final_k]]


post_filter_result = post_filter_search(
    query_vector, doc_vectors_norm, KNOWLEDGE_BASE,
    doc_type=TARGET_DOC_TYPE, top_k_before_filter=2, final_k=3
)
pre_filter_result = pre_filter_search(
    query_vector, doc_vectors_norm, KNOWLEDGE_BASE,
    doc_type=TARGET_DOC_TYPE, final_k=3
)

print("=" * 70)
print(f"POST-FILTER vs PRE-FILTER (target doc_type='{TARGET_DOC_TYPE}', requesting top-3)")
print("=" * 70)
print(f"Query: '{QUERY}'\n")

print(f"POST-FILTER result ({len(post_filter_result)} of 3 requested):")
if not post_filter_result:
    print("  ZERO RESULTS -- even though matching SOP documents exist in the corpus!")
for idx in post_filter_result:
    print(f"  Doc {idx} | {KNOWLEDGE_BASE[idx]['text'][:60]}...")

print(f"\nPRE-FILTER result ({len(pre_filter_result)} of 3 requested):")
for idx in pre_filter_result:
    print(f"  Doc {idx} | {KNOWLEDGE_BASE[idx]['text'][:60]}...")

if len(post_filter_result) < len(pre_filter_result):
    print(f"\nCONFIRMED: post-filter returned {len(post_filter_result)} results,")
    print(f"pre-filter correctly returned {len(pre_filter_result)} -- because only 1 of")
    print("the top-2 overall results (before filtering) happened to be an SOP")
    print("document, even though 2 genuinely relevant SOP documents exist in")
    print("the corpus. This is the exact failure mode described in the theory,")
    print("reproduced with real numbers, not just asserted.")
else:
    print("\nBoth methods returned the same count this time -- try a stricter")
    print("top_k_before_filter value in Module 2 to reproduce the failure")
    print("more dramatically (the smaller this is, the more likely post-filter")
    print("misses matching documents entirely).")

print("\nModule 2 complete. Run Module 3 next.")


POST-FILTER vs PRE-FILTER (target doc_type='sop', requesting top-3)
Query: 'what do I need to submit to close my FD'

POST-FILTER result (1 of 3 requested):
  Doc 4 | Step 1 of FD closure: submit a written closure request form ...

PRE-FILTER result (2 of 3 requested):
  Doc 4 | Step 1 of FD closure: submit a written closure request form ...
  Doc 5 | Step 2 of FD closure: attach a copy of the passbook and iden...

CONFIRMED: post-filter returned 1 results,
pre-filter correctly returned 2 -- because only 1 of
the top-2 overall results (before filtering) happened to be an SOP
document, even though 2 genuinely relevant SOP documents exist in
the corpus. This is the exact failure mode described in the theory,
reproduced with real numbers, not just asserted.

Module 2 complete. Run Module 3 next.


### Module 3: Consistent Filtering Across a Hybrid Retrieval Setup

Demonstrates why applying a filter to only ONE of two retrievers being fused produces inconsistent, incorrect results.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Consistent filtering in hybrid retrieval
#
# Reuses the RRF fusion logic from Topic 4 to show the concrete bug:
# filtering only the dense retriever but not BM25 lets a wrong-category
# document sneak into the final fused result through the unfiltered path.
# ------------------------------------------------------------------

from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    return text.lower().split()

def reciprocal_rank_fusion(ranked_lists: list, k: int = 60) -> dict:
    rrf_scores = {}
    for ranked_list in ranked_lists:
        for position, doc_id in enumerate(ranked_list, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + position)
    return rrf_scores


# dense retrieval: CORRECTLY filtered to sop only
dense_filtered_ranking = pre_filter_search(query_vector, doc_vectors_norm, KNOWLEDGE_BASE,
                                            doc_type=TARGET_DOC_TYPE, final_k=len(KNOWLEDGE_BASE))

# BM25: run WITHOUT any filter applied (the bug being demonstrated)
all_tokenized = [tokenize(d["text"]) for d in KNOWLEDGE_BASE]
bm25_unfiltered = BM25Okapi(all_tokenized)
bm25_scores_all = bm25_unfiltered.get_scores(tokenize(QUERY))
bm25_unfiltered_ranking = list(np.argsort(bm25_scores_all)[::-1])

# BM25: run WITH the SAME filter applied consistently (the fix)
sop_docs_only = [d for d in KNOWLEDGE_BASE if d["doc_type"] == TARGET_DOC_TYPE]
sop_doc_ids = [d["id"] for d in sop_docs_only]
tokenized_sop_only = [tokenize(d["text"]) for d in sop_docs_only]
bm25_filtered_index = BM25Okapi(tokenized_sop_only)
bm25_scores_filtered = bm25_filtered_index.get_scores(tokenize(QUERY))
bm25_filtered_ranking = [sop_doc_ids[i] for i in np.argsort(bm25_scores_filtered)[::-1]]

print("=" * 70)
print("BUG: FILTERING ONLY ONE RETRIEVER IN A HYBRID SETUP")
print("=" * 70)
print(f"Query: '{QUERY}' | target doc_type='{TARGET_DOC_TYPE}'\n")

# INCONSISTENT fusion: dense is filtered, BM25 is not
inconsistent_fused = reciprocal_rank_fusion([dense_filtered_ranking, bm25_unfiltered_ranking])
inconsistent_top = sorted(inconsistent_fused.keys(), key=lambda d: inconsistent_fused[d], reverse=True)[:3]

# CONSISTENT fusion: both filtered identically
consistent_fused = reciprocal_rank_fusion([dense_filtered_ranking, bm25_filtered_ranking])
consistent_top = sorted(consistent_fused.keys(), key=lambda d: consistent_fused[d], reverse=True)[:3]

print("INCONSISTENT fusion (dense filtered, BM25 NOT filtered) -- top 3:")
for idx in inconsistent_top:
    doc = KNOWLEDGE_BASE[idx]
    flag = "  <-- WRONG doc_type leaked through the unfiltered BM25 path!" if doc["doc_type"] != TARGET_DOC_TYPE else ""
    print(f"  Doc {idx} [{doc['doc_type']:<6}] | {doc['text'][:55]}...{flag}")

print("\nCONSISTENT fusion (both retrievers filtered the SAME way) -- top 3:")
for idx in consistent_top:
    doc = KNOWLEDGE_BASE[idx]
    print(f"  Doc {idx} [{doc['doc_type']:<6}] | {doc['text'][:55]}...")

leaked = [i for i in inconsistent_top if KNOWLEDGE_BASE[i]["doc_type"] != TARGET_DOC_TYPE]
if leaked:
    print(f"\nCONFIRMED: doc(s) {leaked} with the WRONG doc_type appeared in the")
    print("inconsistently-filtered fusion result, purely because BM25's path")
    print("was never filtered. In an access-control context, this exact bug")
    print("pattern is how unauthorized records leak through a hybrid system.")
else:
    print("\nNo wrong-category leak in this particular run -- the risk is still")
    print("real and depends on the specific corpus and query; the fix (filter")
    print("BOTH retrievers identically) removes the risk regardless.")

print("\nModule 3 complete. All Topic 8 modules done.")
print()
print("=" * 70)
print("CHAPTER 7 TOPIC 8 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Pre-filter (during search) is correct for almost all cases.
  Post-filter (after search) can silently return fewer results than
  requested, or zero, even when matching documents genuinely exist.

  In hybrid retrieval, the SAME filter must be applied to EVERY
  retriever before fusion -- filtering inconsistently is a correctness
  bug, and a real security risk in access-control contexts.

  Filtering trades recall-if-wrong for precision-if-right -- a highly
  selective filter based on an uncertain signal can eliminate the
  correct answer entirely if that signal turns out to be mistaken.
""")


BUG: FILTERING ONLY ONE RETRIEVER IN A HYBRID SETUP
Query: 'what do I need to submit to close my FD' | target doc_type='sop'

INCONSISTENT fusion (dense filtered, BM25 NOT filtered) -- top 3:
  Doc 4 [sop   ] | Step 1 of FD closure: submit a written closure request ...
  Doc 5 [sop   ] | Step 2 of FD closure: attach a copy of the passbook and...
  Doc 3 [faq   ] | Can I submit a request to close my FD account online in...  <-- WRONG doc_type leaked through the unfiltered BM25 path!

CONSISTENT fusion (both retrievers filtered the SAME way) -- top 3:
  Doc 4 [sop   ] | Step 1 of FD closure: submit a written closure request ...
  Doc 5 [sop   ] | Step 2 of FD closure: attach a copy of the passbook and...

CONFIRMED: doc(s) [np.int64(3)] with the WRONG doc_type appeared in the
inconsistently-filtered fusion result, purely because BM25's path
was never filtered. In an access-control context, this exact bug
pattern is how unauthorized records leak through a hybrid system.

Module 3 complete